In [4]:
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password="password",
    host="localhost",
    port=5555,
    database="similarity_search_service_db"
)

In [5]:
from typing import List
from pgvector.sqlalchemy import Vector
from sqlalchemy import Integer, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


# Create the base class for the table definition
class Base(DeclarativeBase):
    __abstract__ = True


# Create the table definition
class Images(Base):
    __tablename__ = "images"
    VECTOR_LENGTH = 512

    # primary key
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    # image path - we will use it to store the path to the image file, after similarity search we can use it to retrieve the image and display it
    image_path: Mapped[str] = mapped_column(String(256))
    # image embedding - we will store the image embedding in this column, the image embedding is a list of 512 floats this is the output of the sentence transformer model
    image_embedding: Mapped[List[float]] = mapped_column(Vector(VECTOR_LENGTH))

In [6]:
from sqlalchemy import create_engine

engine = create_engine(db_url)

In [7]:
Base.metadata.create_all(engine)

In [8]:
from sqlalchemy.orm import Session
from sqlalchemy import Engine, select
import numpy as np


# reusable function to insert data into the table
def insert_image(engine: Engine, image_path: str, image_embedding: list[float]):
    with Session(engine) as session:
        # create the image object
        image = Images(
            image_path=image_path,
            image_embedding=image_embedding
        )
        # add the image object to the session
        session.add(image)
        # commit the transaction
        session.commit()


# insert some data into the table
N = 100
for i in range(N):
    image_path = f"image_{i}.jpg"
    image_embedding = np.random.rand(512).tolist()
    insert_image(engine, image_path, image_embedding)

# select first image from the table
with Session(engine) as session:
    image = session.query(Images).first()


# Note: pgvector exposes *distances*, not similarities (lower = more similar).
# To rank, sort ascending by `cosine_distance`. To filter by a similarity threshold,
# use `1 - cosine_distance > threshold`.

# order the K images by ascending cosine distance to the first image (smallest distance = most similar)
def find_k_images(engine: Engine, k: int, original_image: Images) -> list[Images]:
    with Session(engine) as session:
        # execution_options={"prebuffer_rows": True} is used to prebuffer the rows, this is useful when we want to fetch the rows in chunks and return them after session is closed
        result = session.execute(
            select(Images)
            .order_by(Images.image_embedding.cosine_distance(original_image.image_embedding))
            .limit(k),
            execution_options={"prebuffer_rows": True}
        ).scalars().all()
        return result


# find the 10 most similar images to the first image
k = 10
similar_images = find_k_images(engine, k, image)

In [9]:
# find images with similarity to the original above the threshold (similarity = 1 - cosine_distance)
def find_images_with_similarity_score_greater_than(engine: Engine, similarity_score: float, original_image: Images) -> \
list[Images]:
    with Session(engine) as session:
        result = session.execute(
            select(Images)
            .filter((1 - Images.image_embedding.cosine_distance(original_image.image_embedding)) > similarity_score),
            execution_options={"prebuffer_rows": True}
        ).scalars().all()
        return result

In [10]:
from datasets import load_dataset

dataset = load_dataset("FronkonGames/steam-games-dataset")

# get columns names and types
columns = dataset["train"].features
print(columns)

columns_to_keep = ["name", "windows", "linux", "mac", "detailed_description", "supported_languages", "price"]

N = 20000  # you can adjust this number
dataset = dataset["train"].select_columns(columns_to_keep).select(range(N))

{'appID': Value('string'), 'name': Value('string'), 'release_date': Value('string'), 'estimated_owners': Value('string'), 'peak_ccu': Value('int64'), 'required_age': Value('int64'), 'price': Value('float64'), 'dlc_count': Value('int64'), 'detailed_description': Value('string'), 'short_description': Value('string'), 'supported_languages': List(Value('string')), 'full_audio_languages': List(Value('string')), 'reviews': Value('string'), 'header_image': Value('string'), 'website': Value('string'), 'support_url': Value('string'), 'support_email': Value('string'), 'windows': Value('bool'), 'mac': Value('bool'), 'linux': Value('bool'), 'metacritic_score': Value('int64'), 'metacritic_url': Value('string'), 'user_score': Value('int64'), 'positive': Value('int64'), 'negative': Value('int64'), 'score_rank': Value('string'), 'achievements': Value('int64'), 'recommendations': Value('int64'), 'notes': Value('string'), 'average_playtime_forever': Value('int64'), 'average_playtime_2weeks': Value('int6

In [11]:
from sqlalchemy import Integer, Float, Boolean, Text


class Games(Base):
    __tablename__ = "games"
    __table_args__ = {'extend_existing': True}

    # the vector size produced by the model taken from documentation https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2
    VECTOR_LENGTH = 512  # check the model output dimensionality

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(256))
    # `Text` is an unbounded string in Postgres — game descriptions can run a few KB
    description: Mapped[str] = mapped_column(Text)
    windows: Mapped[bool] = mapped_column(Boolean)
    linux: Mapped[bool] = mapped_column(Boolean)
    mac: Mapped[bool] = mapped_column(Boolean)
    price: Mapped[float] = mapped_column(Float)
    game_description_embedding: Mapped[List[float]] = mapped_column(
        Vector(VECTOR_LENGTH)
    )  # fill it in proper way


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

In [12]:
from sentence_transformers import SentenceTransformer


checkpoint = "distiluse-base-multilingual-cased-v2"
model = SentenceTransformer(checkpoint)


def generate_embeddings(text: str) -> list[float]:
    return model.encode(text)

In [13]:
from tqdm import tqdm


def insert_games(engine, dataset):
    with tqdm(total=len(dataset)) as pbar:
       for i, game in enumerate(dataset):
           game_description = game["detailed_description"] or ""
           game_embedding = generate_embeddings(game_description)
           name, windows, linux, mac, price = game["name"], game["windows"], game["linux"], game["mac"], game["price"]
           # keep rows with the required fields; windows/linux/mac are platform flags (True/False),
           # not missing-data signals — don't gate the insert on them or you discard most of the dataset
           if name and game_description and price is not None:
               game = Games(
                   name=game["name"],
                   description=game_description,
                   windows=game["windows"],
                   linux=game["linux"],
                   mac=game["mac"],
                   price=game["price"],
                   game_description_embedding=game_embedding
               )
               with Session(engine) as session:
                   session.add(game)
                   session.commit()
           pbar.update(1)

In [14]:
insert_games(engine, dataset)

100%|██████████| 20000/20000 [2:11:53<00:00,  2.53it/s]  


In [16]:
from typing import Optional

def find_game(
    engine: Engine,
    game_description: str,
    windows: Optional[bool] = None,
    linux: Optional[bool] = None,
    mac: Optional[bool] = None,
    price: Optional[int] = None
):
    with Session(engine) as session:
        game_embedding = generate_embeddings(game_description)# generate game embedding

        query = (
            select(Games)
            .order_by(Games.game_description_embedding.cosine_distance(game_embedding))
        )

        if price:
            query = query.filter(Games.price <= price)
        if windows:
            query = query.filter(Games.windows == True)
        if linux:
            query = query.filter(Games.linux == True)
        if mac:
            query = query.filter(Games.mac == True)

        result = session.execute(query, execution_options={"prebuffer_rows": True})
        game = result.scalars().first()

        return game

In [17]:
game = find_game(engine, "This is a game about a hero who saves the world", price=10)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

game = find_game(engine, game_description="Home decorating", price=20)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: FadeZone 消逝之地
Description: It's an adventure role-playing game, you've been summoned to this world as a Hero, along the way you will encounter various puzzles and NPCs, and eventually defeat Mayor(the mayor) assimilated with the devil, save this world. * But, only half of this description is truth. If someone wanna translate this game to other language. You are welcome! ——————————Spoilers Warning—————————— Players will control a silent, thoughtless, weakness 'Hero' .Come to the town. With the Black cat, complete the puzzles, and kill the Devil of Mayor. But, it is not the end. Players can get more stories and learn the truth of the world in the playing again and again. The so-called 'Hero fight against the Devil' is only a forced disguise, hidden behind the disguise is indeed conspiracy and rights? And the truth of everything will be very cruel, so-called rights are useless, so-called conspiracy are foolish disguises. 'The world is just a box. Inside box, there's another box. Box

In [18]:
game = find_game(engine, game_description="Home decorating", mac=True, price=5)
print(f"Game: {game.name}")
print(f"Description: {game.description}")

Game: Clown House (Palyaço Evi)
Description: Clown House is a short, unique horror experience. You must find the key to escape from the Clown House. Some of the residents are harmless, some of them are not. You carry a gun, but you can't know which clowns are willing to harm you unless they start to attack. Worst part? Killing an innocent clown turns you into one of them. 


Retrieval-Augmented Generation (RAG) service

In [19]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(
    host=host,
    port=port
)

In [20]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")

In [21]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding",
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
)

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 466783791860166216, 'consistency_level': 2, 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 466784347445526545, 'update_timestamp': 466784347445526545}


In [22]:
# define data source and destination
## the document origin destination from which document will be downloaded
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [23]:
# download data
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)

In [24]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)

In [26]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))

        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [28]:
# search

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings",
        data=[embedded_query],
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [32]:
import os
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-2.5-flash"

def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [49]:
def build_prompt(context: str, query: str) -> str:
    prompt = f"""
You are a MLOps help assistant.

Answer the question using the context provided below.
If the context does not contain a full answer, try to answer as best as you can based on the available information.


Context:
{context}

Question:
{query}

Answer:
"""# try to build the prompt by yourself!
    return prompt


def rag(model, query: str) -> str:
    result = search(model, query)

    context = "\n\n".join(
        hit.entity.text for hit in result[0]
    )

    prompt = build_prompt(context, query)
    return generate_response(prompt)
    # having all prepared functions, you can combine them together and try to build your own RAG!

In [50]:
print(rag(model, "Czym jest sztuczna inteligencja"))

Sztuczna inteligencja (AI) to:

*   W potocznym rozumieniu: coś (program, maszyna) symulującego inteligencję naturalną, ludzką.
*   Obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań, które wymagają ludzkiej inteligencji.
*   System, którego głównym celem jest zdolność do myślenia i podejmowania decyzji w sposób przypominający ludzki, a także realizowanie niektórych funkcji umysłu. Czasami ma ona przewyższać funkcje naturalne, np. być wolna od pomyłek w liczeniu czy defektów pamięci.

Zadania, które obejmuje sztuczna inteligencja, to m.in. rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się i planowanie.


In [51]:
print(rag(model, "Jakie są rodzaje sztucznej inteligencji?"))

Na podstawie dostarczonego kontekstu, główne rodzaje sztucznej inteligencji to:

*   **Sztuczna inteligencja słaba (wąska)**
*   **Sztuczna inteligencja silna**
*   **Sztuczna inteligencja umysłowa**
*   **Sztuczna inteligencja zwiększająca (Augmented Intelligence)**
*   **Sztuczna inteligencja bezpieczna (Safe AI)**
*   **Sztuczna inteligencja odwrócona (Inverse AI)**
*   **Sztuczna inteligencja reprezentacyjna (Representational AI)**
*   **Sztuczna inteligencja predykcyjna (Predictive AI)**


In [52]:
print(rag(model, "Kim jest Maciej Leonard Żybula?"))

Maciej Leonard Żybula jest autorem lub prelegentem materiału o tytule "WPROWADZENIE DO AI".


Tests

Style: force pirate speak, Shakespearean English, Yoda, or corporate jargon.

In [55]:
def style_prompt(query: str, style: str) -> str:
    return f"""
You are a MLOps help assistant.

Answer the question using the provided context.
IMPORTANT: respond in {style} style.

Styles:
- pirate: pirate speak
- shakespearean: Shakespearean English
- yoda: Yoda-like syntax
- corporate: corporate/business jargon

Question:
{query}

Answer:
"""

In [57]:
print(rag(model, style_prompt("Co to jest Sztuczna inteligencja silna?", "pirate")))

Arrr, me heartie! Ye be askin' about 'Sztuczna inteligencja silna' (Strong AI), eh?

Blow me down, the parchment o' knowledge I hold doesn't explicitly chart the definition o' 'Sztuczna inteligencja silna'!

But fear not, I can tell ye what 'Sztuczna inteligencja' (AI, for short, ya scallywag!) generally be: 'tis systems or machines that ape human smarts whilst doin' their duties. And there be a kind called 'wąska sztuczna inteligencja' – that's when a learnin' machine gets clever enough at one specific task, quick and true, to be mighty useful and trusted!

So, while the 'strong' kind ain't fully charted in these waters, ye've got the general idea and a bit o' the 'narrow' kind to mull over, savvy?


In [59]:
print(rag(model, style_prompt("Co to jest Sztuczna inteligencja odwrócona?", "yoda")))

Ludzkich procesów myślowych modelowaniem, Sztuczna inteligencja odwrócona zajmuje się. Lepsze zrozumienie i dostosowanie interakcji maszyn do ludzkich oczekiwań, jej to cel jest.


In [61]:
print(rag(model, style_prompt("Co to jest Sztuczna inteligencja umysłowa?", "shakespearean")))

Hark, seeker of knowledge! Thou dost inquire of "Sztuczna inteligencja umysłowa," yet in the scrolls before mine eyes, no distinct definition for this precise phrase doth appear.

Nonetheless, of Artificial Intelligence – which doth mimic the very essence of human wit – 'tis plainly writ that it be systems or machines fashioned to emulate man's intelligence whilst they perform their sundry tasks. Verily, these be programs capable of learning and reasoning, much as a human mind doth apprehend and deduce. Perchance, by "umysłowa" (or "mental"), thou dost speak of this very imitation of human thought and reason, which lies at the heart of all Artificial Intelligence.


In [62]:
print(rag(model, style_prompt("Co to jest Sztuczna inteligencja reprezentacyjna?", "corporate")))

Based on the provided context, the term "Sztuczna inteligencja reprezentacyjna" is not explicitly defined.

However, the context extensively defines **Sztuczna inteligencja (AI)**. From a corporate perspective, Artificial Intelligence (AI) can be understood as a domain within computer science dedicated to the development of sophisticated computer programs and systems. These systems are engineered to perform tasks traditionally requiring human intelligence, such as pattern recognition, natural language comprehension, strategic decision-making, continuous learning, and advanced planning. The overarching objective of AI is to establish systems capable of cognitive processes and decision-making that emulate or, in certain specific functions, even surpass human capabilities, for instance, in terms of computational accuracy or memory retention.


Language: instruct the model to answer in Polish (or Japanese, Latin, …) even though the retrieved context is in English.

In [63]:
def language_prompt(query: str, language: str) -> str:
    return f"""
You are a MLOps help assistant.

Answer the question using the context provided.
IMPORTANT: Respond in {language}, even if the context is in English.

Question:
{query}

Answer:
"""

In [64]:
print(rag(model, language_prompt("What is Machine Learning?", "Polish")))

Uczenie maszynowe to proces, który systemy komputerowe wykonują, aby osiągnąć sztuczną inteligencję. Wykorzystuje on algorytmy do identyfikowania wzorców w danych, które są następnie używane do utworzenia modelu danych umożliwiającego przewidywanie. Jest to również definiowane jako algorytmy zdolne do uczenia się bez bezpośredniego programowania.


In [65]:
print(rag(model, language_prompt("What is Deep Learning?", "Japanese")))

ディープラーニング（深層学習）は、機械学習のサブセットであり、人工ニューラルネットワークが膨大な量のデータに基づいて適応し、学習するものです。これは、脳の構造に触発されたアルゴリズムのネットワークであるニューラルネットワークを利用する、機械学習の高度なタイプです。通常、トレーニングには数百万のデータポイントからなる大規模なデータセットが必要となります。


In [73]:
print(rag(model, language_prompt("What are reactive machines?", "Latin")))

Machinae reactivae sunt Intelligentia Artificialis creata ad problema specificum solvendum. Memoria carent, et ideo discere non possunt. Exemplum est Deep Blue, programma scaccorum IBM, quod Garry Kasparov in annis nonaginta vicit.


Format: force a haiku, a single sentence, exactly three bullet points, or "emoji + one-line explanation" per fact.

In [74]:
def format_prompt(query: str, response_format: str) -> str:
    return f"""
You are a MLOps help assistant.

Answer the question using the provided context.
IMPORTANT: Format your answer as {response_format}.

Question:
{query}

Answer ({response_format}):
"""

In [75]:
print(rag(model, format_prompt("What are expert systems?", "haiku")))

Error generating response: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 26.84687104s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini

In [ ]:
print(rag(model, format_prompt("What are Neural networks?", "single sentence")))

In [ ]:
print(rag(model, format_prompt("What is Data mining?", "three bullet points")))

In [ ]:
print(rag(model, format_prompt("What is image recognition?", "emoji + one-line explanation")))

Persona: a grumpy professor; an over-enthusiastic YouTuber; a skeptical reviewer that grades each claim "supported / unsupported by context".

In [ ]:
def persona_prompt(query: str, persona: str) -> str:
    return f"""
You are a MLOps help assistant.

Answer the question using the provided context.
IMPORTANT: Respond as the following persona: {persona}.

If the persona is "skeptical reviewer", you MUST evaluate each claim and label it as:
- supported by context
- unsupported by context

Question:
{query}

Answer:
"""

In [ ]:
print(rag(model, persona_prompt("What is OCR?", "grumpy professor")))

In [ ]:
print(rag(model, persona_prompt("What is image generation?", "over-enthusiastic YouTuber")))

In [ ]:
print(rag(model, persona_prompt("What is Generative Adversarial Networks?", "skeptical reviewer")))

Constraints: answer without using the letter "e"; max 50 words; each sentence starts with the next letter of the alphabet.

In [ ]:
def constraints_prompt(query: str, constraints: str) -> str:
    return f"""
You are a MLOps help assistant.

Answer the question using the provided context.
IMPORTANT: Follow these constraints strictly: {constraints}

Question:
{query}

Answer:
"""

In [ ]:
print(rag(model, constraints_prompt("What is WaveGAN?", "answer without using the letter 'e'")))

In [ ]:
print(rag(model, constraints_prompt("What is Deepfake", "max 50 words")))

In [ ]:
print(rag(model, constraints_prompt("What is Pix2Pix?", "each sentence starts with the next letter of the alphabet")))

Hallucination probe: ask something not in the retrieved context and check whether the model invents an answer or honestly says "not in sources". A good RAG prompt should make refusal easy.

In [ ]:
print(rag(model, "Kim jest Maria Skłodowska-Curie?"))